In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader
class SubtractOne(nn.Module):
  def forward(self, img):
    return img-1
# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    # TODO: Resize to 28x28
    transforms.Resize((28,28)),
    transforms.Grayscale(3),  # Convert grayscale to RGB (Don't Touch!!)
    # TODO: Convert to Tensor
    transforms.ToTensor(),
    # TODO: Normalize with ImageNet mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225]
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    SubtractOne(),

])

# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'
classL = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F":6, "G":7, "H":8, "I":9,
                "J":10, "K":11, "L":12, "M":13, "N":14,"O":15, "P":16, "Q":17, "R":18, "S":19, "T":20 ,"U":21, "V":22, "W":23, "X":24, "Y":25, "Z":26 }

# Create DataLoaders and display samples
train=DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
test = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)
# Write your code here
import matplotlib.pyplot as plt
import numpy as np

# Get a batch of training data
data_iter = iter(train)
images,_= next(data_iter)  # Labels are ignored in Autoencoder

# Show images
fig, axes = plt.subplots(2, 5, figsize=(10, 5))
for i, ax in enumerate(axes.flat):


    img = images[i].squeeze(0)
    img = np.transpose(img.numpy(), (1, 2, 0))  # Remove channel dimension (1, 28, 28) → (28, 28)

    ax.imshow(img)

    ax.axis("off")

plt.show()

# Show shape of one image
print("Shape of one image tensor:", images[0].shape)  # Expected: (1, 28, 28)


In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s

# Write your code here
from torchvision import models

model = models.efficientnet_v2_s(pretrained=True)
for param in model.parameters():
    param.requires_grad = False

# Replace classifier head (this will be trainable by default)
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 26)  # Binary classification

# Move to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Verify what's trainable
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Training {trainable_params:,} / {total_params:,} parameters ({100*trainable_params/total_params:.2f}%)")

In [ ]:
import torch.optim as optim

# Write your code here
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.classifier.parameters(), lr=0.001)

def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train() # Set the model to training mode
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad() # Zero the parameter gradients

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()


    epoch_loss = running_loss / total_samples
    epoch_accuracy = correct_predictions / total_samples
    return epoch_loss, epoch_accuracy

def validate_epoch(model, dataloader, criterion, device):
      model.eval() # Set the model to evaluation mode
      running_loss = 0.0
      correct_predictions = 0
      total_samples = 0

      with torch.no_grad(): # Disable gradient calculation
        for inputs, labels in dataloader:
              inputs, labels = inputs.to(device), labels.to(device)

              outputs = model(inputs)
              loss = criterion(outputs, labels)

              running_loss += loss.item() * inputs.size(0)
              _, predicted = torch.max(outputs.data, 1)
              total_samples += labels.size(0)
              correct_predictions += (predicted == labels).sum().item()

        epoch_loss = running_loss / total_samples
        epoch_accuracy = correct_predictions / total_samples
        return epoch_loss, epoch_accuracy


In [ ]:
num_epochs = 5

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

print("Starting Training...")
for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train, criterion, optimizer, device)
    val_loss, val_acc = validate_epoch(model, test, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(f'Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

print("Finished Training.")


In [ ]:
# Write your code here
def validate_epoch(model, dataloader, criterion, device):
      model.eval() # Set the model to evaluation mode
      running_loss = 0.0
      correct_predictions = 0
      total_samples = 0

      with torch.no_grad(): # Disable gradient calculation
        for inputs, labels in dataloader:
              inputs, labels = inputs.to(device), labels.to(device)

              output1 = model(inputs)
              output2 = model(inputs,h_flipped = torch.flip(images, dims=[3]))
              output3 = model(v_flipped = torch.flip(images, dims=[2]))
              outputs=(output1+output2+output3)/3
              loss = criterion(outputs, labels)

              running_loss += loss.item() * inputs.size(0)
              _, predicted = torch.max(outputs.data, 1)
              total_samples += labels.size(0)
              correct_predictions += (predicted == labels).sum().item()

        epoch_loss = running_loss / total_samples
        epoch_accuracy = correct_predictions / total_samples
        return epoch_loss, epoch_accuracy
